In [1]:
import sys
import math
import os
import torch
sys.path.append(os.path.abspath('../'))

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from torch_geometric.data import Data
from tqdm import tqdm

# Add both project root and src/ to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))  # <-- add this

from src.dataset import deterministic_sample
from src.egnn import EGNN
from src.fm import FlowMatching
from src.pbc_config import wrap, min_image, BOX
from src.utils import load_config, gpu_knn_graph_pbc_batch, scale_thetas

from src.validation import get_tpcf

import numpy as np
import pandas as pd 
import torch.nn as nn
import matplotlib.pyplot as plt

/home/bartb/venvs/boids/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_run = "20260311_020731"
config_dir = f"/gpfs/home4/bartb/T5000/results/run_{train_run}/train_configs.yaml"
config = load_config(config_dir)

In [ ]:
egnn = EGNN(t_embed_dim=config["model"]["fm"]["t_embed_dim"],
                input_node_d=config["model"]["egnn"]["input_node_d"],
                input_theta_d=5,
                theta_param_embd_dim=config["model"]["egnn"]["theta_param_embd_dim"],
                hidden_nf=config["model"]["egnn"]["hidden_nf"],
                latent_nf=config["model"]["egnn"]["latent_nf"],
                theta_nf=config["model"]["egnn"]["theta_nf"],
                n_layers=config["model"]["egnn"]["n_layers"],
                mlp_layers=config["model"]["egnn"]["mlp_layers"],
                single_layer=config["model"]["egnn"]["single_layer"],
                recurrent=config["model"]["egnn"]["recurrent"],
                activation=nn.SiLU(),
                norm=config["model"]["egnn"]["norm"],
                attention=config["model"]["egnn"]["attention"],
                scale_pred=config["model"]["egnn"]["scale_pred"],
                coords_weight=config["model"]["egnn"]["coords_weight"],
                norm_diff=config["model"]["egnn"]["norm_diff"])
    
model = FlowMatching(sigma_0=config["model"]["fm"]["sigma_0"],
                     sigma_sched=config["model"]["fm"]["sigma_sched"],
                     t_embed_dim=config["model"]["fm"]["t_embed_dim"],
                     version=config["model"]["fm"]["version"],
                     vnet=egnn,
                     batch_size=config["training"]["batch_size"],
                     prior=config["model"]["fm"]["prior"],
                     k=config["model"]["fm"]["k"],
                     t_embed=config["model"]["fm"]["t_embed"],
                     n_halos=config["model"]["fm"]["n_halos"],
                     dim=3
                     )

checkpoint = torch.load("/gpfs/home4/bartb/T5000/results/run_20260311_020731/model_epoch_1400.pth", map_location=torch.device('cpu'))
# checkpoint = torch.load(f"/gpfs/home4/bartb/T5000/results/run_{train_run}/model_final.pth", map_location=torch.device('cpu'))
new_state_dict = {k.replace("module.", ""): v for k, v in checkpoint["model_state"].items()}  # Remove "module."
model.load_state_dict(new_state_dict)

print("Loaded model")